# Description
We aim to compare the process of creating a PD model using a logistic regression and a common ML method (LGBM - Ligh Gradient Boosting Machine)
This notebook will train the logistic regression model.

# Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import Image
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
import statsmodels.api as sm
import joblib

In [2]:
%run Utils.ipynb

# Data

In [3]:
project_folder = "02_base_pipeline"

input_path = f"..\\..\\data\\inputs\\{project_folder}\\"
output_path = f"..\\..\\data\\outputs\\{project_folder}\\"

In [4]:
train_df = pd.read_parquet(f'{output_path}train_df.parquet')
test_df  = pd.read_parquet(f'{output_path}test_df.parquet')

In [5]:
cat_binner = joblib.load(f'{output_path}cat_binner.gz')

In [6]:
train_df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,verification_status_bands,verification_status_woe,dti_bands,dti_woe,fico_range_low_bands,fico_range_low_woe,inq_last_6mths_bands,inq_last_6mths_woe,revol_util_bands,revol_util_woe
0,7400.0,7400.0,7400.0,36 months,0.1825,268.46,E,E1,10+ years,MORTGAGE,...,1,-0.403293,5,0.339305,14,-0.514567,0,-0.134986,0,-0.456498
1,6000.0,6000.0,6000.0,36 months,0.0958,192.43,B,B1,None,MORTGAGE,...,0,0.078777,5,0.339305,12,-0.381965,1,0.083006,0,-0.456498
2,8500.0,8500.0,8500.0,36 months,0.0699,262.42,A,A2,10+ years,MORTGAGE,...,1,-0.403293,3,0.002166,7,-0.019309,1,0.083006,3,-0.062727
3,20000.0,20000.0,20000.0,36 months,0.1049,649.96,B,B3,10+ years,MORTGAGE,...,1,-0.403293,1,-0.248982,6,0.062224,0,-0.134986,1,-0.350623
4,15000.0,15000.0,15000.0,36 months,0.1099,491.01,B,B4,2 years,MORTGAGE,...,1,-0.403293,3,0.002166,11,-0.318577,0,-0.134986,5,0.103752


# Analysis

## Log reg

In [7]:
x_cols = [f'{col_i}_woe' for col_i in cat_binner.selected_cols]
y_col = ['default_flag']

Xtrain = train_df[x_cols]
ytrain = train_df[y_col]

Xtest = test_df[x_cols]
ytest = test_df[y_col]

In [8]:
def stepwise_selection(X, y,
                       initial_list=[],
                       threshold_in=0.01,
                       threshold_out = 0.05,
                       max_features = 30,
                       verbose=True):
    """ Perform a forward-backward feature selection
    based on p-value from statsmodels.api.OLS
    Arguments:
        X - pandas.DataFrame with candidate features
        y - list-like with the target
        initial_list - list of features to start with (column names of X)
        threshold_in - include a feature if its p-value < threshold_in
        threshold_out - exclude a feature if its p-value > threshold_out
        verbose - whether to print the sequence of inclusions and exclusions
    Returns: list of selected features
    Always set threshold_in < threshold_out to avoid infinite looping.
    See https://en.wikipedia.org/wiki/Stepwise_regression for the details
    """
    included = list(X.columns)
    while True:
        changed=False
        # forward step
        excluded = list(set(X.columns)-set(included))
        # new_pval = pd.Series(index=excluded)
        # for new_column in excluded:
        #     model = sm.GLM(y, sm.add_constant(pd.DataFrame(X[included+[new_column]])), family=sm.families.Binomial()).fit()
        #     new_pval[new_column] = model.pvalues[new_column]
        # best_pval = new_pval.min()
        # if best_pval < threshold_in:
        #     best_feature = list(new_pval.index)[new_pval.argmin()]
        #     included.append(best_feature)
        #     changed=True
        #     if verbose:
        #         print('Add  {:30} with p-value {:.6}'.format(best_feature, best_pval))

        # backward step
        Xtrain = X[[i for i in included if i not in excluded]]
        Xtrain = sm.add_constant(Xtrain)
        log_reg = sm.Logit(y, Xtrain).fit()
        # use all coefs except intercept
        pvalues = log_reg.pvalues.iloc[1:]
        worst_pval = pvalues.max() # null if pvalues is empty
        if worst_pval > threshold_out:
            changed=True
            worst_feature = list(pvalues.index)[pvalues.argmax()]
            included.remove(worst_feature)
            if verbose:
                print('Drop {:30} with p-value {:.6}'.format(worst_feature, worst_pval))
        if not changed:
            break
        if len(included) >= max_features:
            break
    return included

result = stepwise_selection(Xtrain, ytrain)

print('resulting features:')
print(result)

Optimization terminated successfully.
         Current function value: 0.459864
         Iterations 6
resulting features:
['funded_amnt_woe', 'term_woe', 'sub_grade_woe', 'home_ownership_woe', 'annual_inc_woe', 'verification_status_woe', 'dti_woe', 'fico_range_low_woe', 'inq_last_6mths_woe', 'revol_util_woe']


In [9]:
result = ['funded_amnt_woe', 
          'term_woe', 
          'sub_grade_woe',
          'home_ownership_woe',
          'annual_inc_woe',
          'verification_status_woe',
          'dti_woe',
          'fico_range_low_woe',
          'inq_last_6mths_woe',
          'revol_util_woe']

Xtrain_selected = Xtrain[result]
Xtrain_selected = sm.add_constant(Xtrain_selected)
log_reg = sm.Logit(ytrain, Xtrain_selected).fit()
log_reg.summary()

Optimization terminated successfully.
         Current function value: 0.459864
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:           default_flag   No. Observations:               619801
Model:                          Logit   Df Residuals:                   619790
Method:                           MLE   Df Model:                           10
Date:                Mon, 16 Jun 2025   Pseudo R-squ.:                 0.09052
Time:                        18:32:40   Log-Likelihood:            -2.8502e+05
converged:                       True   LL-Null:                   -3.1339e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
===========================================================================================
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -1.3648      0.003   -402.060      0.000      -1.371      -1.358
funded_amnt_woe             0.6308      0.023     27.217      0.000       0.585       0.676
term_woe                    0.5204      0.010     54.120      0.000       0.502       0.539
sub_grade_woe               0.6292      0.007     95.995      0.000       0.616       0.642
home_ownership_woe          0.8770      0.020     44.716      0.000       0.839       0.915
annual_inc_woe              0.7737      0.022     34.659      0.000       0.730       0.817
verification_status_woe     0.3082      0.014     21.831      0.000       0.281       0.336
dti_woe                     0.5889      0.011     51.971      0.000       0.567       0.611
fico_range_low_woe          0.4899      0.012     41.726      0.000       0.467       0.513
inq_last_6mths_woe          0.4341      0.021     20.469      0.000       0.393       0.476
revol_util_woe             -0.1613      0.021     -7.502      0.000      -0.203      -0.119
===========================================================================================
"""

In [10]:
train_df['pred'] = log_reg.predict(Xtrain_selected)
train_df[['default_flag', 'pred']].agg('mean')

default_flag    0.203806
pred            0.203806
dtype: float64

In [11]:
Xtest_selected = Xtest[result]
Xtest_selected = sm.add_constant(Xtest_selected)

test_df['pred'] = log_reg.predict(Xtest_selected)
test_df[['default_flag', 'pred']].agg('mean')

default_flag    0.202376
pred            0.203708
dtype: float64

In [12]:
joblib.dump(log_reg, f'{output_path}log_reg_out.gz')

['..\\..\\data\\outputs\\02_base_pipeline\\log_reg_out.gz']

## Logistic Regression with regularisation

In [13]:
params = {'penalty': [None, 'l1', 'l2', 'elasticnet'],
          'C': [0.01, 0.05, 0.1, 0.5, 1, 2],
          'l1_ratio': [0.01, 0.05, 0.1, 0.5, 0.75],
          'solver': ['saga']
          }

log_reg_opt = LogisticRegression()

clf = RandomizedSearchCV(log_reg_opt, params, n_iter=150, random_state=42, scoring='neg_log_loss', cv=2)

Xtrain_selected = Xtrain[result]
clf.fit(Xtrain_selected, ytrain)

c:\Users\GL838PA\OneDrive - EY\Documents\amts\amts\.venv\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 120 is smaller than n_iter=150. Running 120 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\GL838PA\OneDrive - EY\Documents\amts\amts\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=None)
  warnings.warn(
c:\Users\GL838PA\OneDrive - EY\Documents\amts\amts\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\GL838PA\OneDrive - EY\Documents\amts\amts\.venv\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=

RandomizedSearchCV(cv=2, estimator=LogisticRegression(), n_iter=150,
                   param_distributions={'C': [0.01, 0.05, 0.1, 0.5, 1, 2],
                                        'l1_ratio': [0.01, 0.05, 0.1, 0.5,
                                                     0.75],
                                        'penalty': [None, 'l1', 'l2',
                                                    'elasticnet'],
                                        'solver': ['saga']},
                   random_state=42, scoring='neg_log_loss')

In [14]:
train_df['pred'] = clf.best_estimator_.predict_proba(Xtrain_selected)[:,1]
train_df[['default_flag', 'pred']].agg('mean')

default_flag    0.203806
pred            0.203805
dtype: float64

In [15]:
Xtest_selected = Xtest[result]

test_df['pred'] = clf.best_estimator_.predict_proba(Xtest_selected)[:,1]
test_df[['default_flag', 'pred']].agg('mean')

default_flag    0.202376
pred            0.203707
dtype: float64

In [16]:
dt_chart = pd.DataFrame(clf.cv_results_).sort_values('rank_test_score')
fig = px.line(dt_chart,
              x='rank_test_score',
              y='mean_test_score',
              template='none',
              width=600)
fig

In [17]:
clf.best_estimator_.feature_names_in_

array(['funded_amnt_woe', 'term_woe', 'sub_grade_woe',
       'home_ownership_woe', 'annual_inc_woe', 'verification_status_woe',
       'dti_woe', 'fico_range_low_woe', 'inq_last_6mths_woe',
       'revol_util_woe'], dtype=object)

In [18]:
joblib.dump(clf.best_estimator_, f'{output_path}log_reg_opt.gz')

['..\\..\\data\\outputs\\02_base_pipeline\\log_reg_opt.gz']